In [ ]:
import sys
from pathlib import Path

# Check notebook working directory
print("Current directory:", Path.cwd())

# Locate backend directory
backend_path = Path.cwd().resolve().parent

# Add backend directory to Python's import path
if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print("Backend path:", backend_path)

In [ ]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

In [ ]:
from app.rag.ingestion.azure_blob import azure_blob_service
from app.rag.ingestion.loaders import document_loader
from app.rag.ingestion.cleaner import document_cleaner

raw_documents = []

for blob_name in azure_blob_service.list_blobs():
    blob_data = azure_blob_service.download_blob(blob_name)
    raw_documents.extend(document_loader.load(blob_name, blob_data))

clean_documents = document_cleaner.clean(raw_documents)

print("Raw documents:", len(raw_documents))
print("Clean documents:", len(clean_documents))

In [ ]:
from collections import defaultdict

document_lengths = defaultdict(list)

for doc in clean_documents:
    doc_type = doc.metadata["document_type"]
    document_lengths[doc_type].append(len(doc.page_content))

for doc_type, lengths in document_lengths.items():
    print(
        f"{doc_type:25} "
        f"Count: {len(lengths):2} | "
        f"Min: {min(lengths):4} | "
        f"Max: {max(lengths):4} | "
        f"Average: {sum(lengths)/len(lengths):.0f}"
    )

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

long_doc_types = {
    "product_documentation",
    "troubleshooting_manual",
    "release_notes",
    "incident_report"
}

long_documents = [
    doc for doc in clean_documents
    if doc.metadata.get("document_type") in long_doc_types
]

chunks = splitter.split_documents(long_documents)

print("Long-form Documents:", len(long_documents))
print("Chunks created:", len(chunks))

In [ ]:
for i, chunk in enumerate(chunks[:5], start=1):

    print(f"\n{'=' * 60}")
    print(f"CHUNK {i}")
    print("Source:", chunk.metadata["source"])
    print("Length:", len(chunk.page_content))
    print("Content:")
    print(chunk.page_content)

In [ ]:
pdf_doc = next(
    doc for doc in clean_documents
    if doc.metadata["document_type"] == "product_documentation"
)

print(pdf_doc.page_content)

In [ ]:
import re

text = pdf_doc.page_content

sections = re.split(
    r"(?m)(?=^\d+\.\s+[A-Z])",
    text
)

sections = [
    section.strip()
    for section in sections
    if section.strip()
]

for i, section in enumerate(sections, start=1):
    print(f"\nSECTION {i}")
    print("Length:", len(section))
    print("Preview:", section[:150])

In [ ]:
from langchain_core.documents import Document

document_header = sections[0]
technical_sections = sections[1:]

pdf_chunks = []

for index, section in enumerate(technical_sections, start=1):

    chunk_content = (
        f"{document_header}\n\n"
        f"{section}"
    )

    chunk = Document(
        page_content=chunk_content,
        metadata={
            **pdf_doc.metadata,
            "section_number": index,
            "chunk_index": index - 1,
        }
    )

    pdf_chunks.append(chunk)

for chunk in pdf_chunks:
    print(
        f"Section: {chunk.metadata['section_number']} | "
        f"Length: {len(chunk.page_content)}"
    )

In [ ]:
for doc_type in [
    "troubleshooting_manual",
    "release_notes",
    "incident_report"
]:
    doc = next(
        d for d in clean_documents
        if d.metadata["document_type"] == doc_type
    )

    print("\n" + "=" * 70)
    print("DOCUMENT TYPE:", doc_type)
    print("SOURCE:", doc.metadata["source"])
    print("LENGTH:", len(doc.page_content))
    print("=" * 70)
    print(doc.page_content)

In [ ]:
import importlib
import app.rag.ingestion.chunker as chunker_module

# Reload the updated chunker.py file
importlib.reload(chunker_module)

# Create a new object from the updated class
chunker = chunker_module.DocumentChunker()

print("DOCX method available:", hasattr(chunker, "_split_docx_sections"))

In [ ]:
from app.rag.ingestion.chunker import DocumentChunker

chunker = DocumentChunker()

pdf_doc = next(
    doc for doc in clean_documents
    if doc.metadata["document_type"] == "product_documentation"
)

pdf_chunks = chunker._split_pdf_sections(pdf_doc)

print("PDF chunks created:", len(pdf_chunks))

for chunk in pdf_chunks:
    print(
        chunk.metadata["section_title"],
        "| Length:",
        len(chunk.page_content)
    )

In [ ]:
from app.rag.ingestion.chunker import DocumentChunker

chunker = DocumentChunker()

docx_doc = next(
    doc for doc in clean_documents
    if doc.metadata["document_type"] == "troubleshooting_manual"
)

docx_chunks = chunker._split_docx_sections(docx_doc)

print("DOCX chunks created:", len(docx_chunks))

for chunk in docx_chunks:
    print(
        chunk.metadata["section_title"],
        "| Length:",
        len(chunk.page_content)
    )

In [ ]:
import importlib
import app.rag.ingestion.chunker as chunker_module

importlib.reload(chunker_module)

chunker = chunker_module.DocumentChunker()

all_chunks = chunker.chunk(clean_documents)

print("Clean documents:", len(clean_documents))
print("Total chunks:", len(all_chunks))

from collections import Counter

chunk_counts = Counter(
    chunk.metadata.get("document_type")
    for chunk in all_chunks
)

for doc_type, count in chunk_counts.items():
    print(f"{doc_type}: {count}")

In [ ]:
for chunk in all_chunks[:5]:
    print({
        "source": chunk.metadata.get("source"),
        "document_type": chunk.metadata.get("document_type"),
        "section_title": chunk.metadata.get("section_title"),
        "length": len(chunk.page_content)
    })

In [ ]:
chunker = chunker_module.DocumentChunker()

all_chunks = chunker.chunk(clean_documents)

chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in all_chunks
]

print("Total chunks:", len(all_chunks))
print("Unique chunk IDs:", len(set(chunk_ids)))
print("All IDs unique:", len(chunk_ids) == len(set(chunk_ids)))

print("Example chunk ID:", chunk_ids[0])

second_run = chunker.chunk(clean_documents)

print(
    "IDs stable:",
    [c.metadata["chunk_id"] for c in all_chunks]
    ==
    [c.metadata["chunk_id"] for c in second_run]
)

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

sentences = [
    "OAuth authentication fails because of server clock drift.",
    "Users cannot log in because the server time is incorrect.",
    "Customers are receiving duplicate ticket notifications."
]

embeddings = embedding_model.encode(sentences)
print("Shape:", embeddings.shape)
print("First sentence embedding:", embeddings[0][:10])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

print(similarity_matrix)

In [ ]:
chunk_texts = [
    chunk.page_content
    for chunk in all_chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True
)

print("Total chunks:", len(all_chunks))
print("Embedding shape:", chunk_embeddings.shape)

In [ ]:
similarity_matrix_all_chunks = cosine_similarity(chunk_embeddings)

print(similarity_matrix_all_chunks)

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

query = "Why is OAuth authentication failing due to incorrect server time?"

# Convert the user question into an embedding.
query_embedding = embedding_model.encode([query])

# Compare the query against all 43 chunk embeddings.
similarity_scores = cosine_similarity(
    query_embedding,
    chunk_embeddings
)[0]

# Find the indices of the top 3 most similar chunks.
top_k = 3

top_indices = np.argsort(similarity_scores)[::-1][:top_k]

# Display retrieved chunks.
for rank, index in enumerate(top_indices, start=1):

    chunk = all_chunks[index]

    print(f"\n{'=' * 70}")
    print(f"Rank: {rank}")
    print(f"Similarity: {similarity_scores[index]:.4f}")
    print(f"Source: {chunk.metadata.get('source')}")
    print(f"Document Type: {chunk.metadata.get('document_type')}")
    print(f"Section: {chunk.metadata.get('section_title')}")
    print(f"{'=' * 70}")

    print(chunk.page_content)

In [ ]:
import importlib
import app.rag.embeddings.embedding_service as embedding_module

importlib.reload(embedding_module)

service = embedding_module.EmbeddingService()

texts = [chunk.page_content for chunk in all_chunks]

vectors = service.embed_documents(texts)

query_vector = service.embed_query(
    "Why is OAuth authentication failing due to server time drift?"
)

print("Document embeddings:", vectors.shape)
print("Query embedding:", query_vector.shape)
print("Embedding dimension:", service.dimension)

In [ ]:
import importlib
import app.rag.vectorstore.pinecone_store as pinecone_module

importlib.reload(pinecone_module)

store = pinecone_module.PineconeStore()

stats = store.check_connection()

print("Index dimension:", stats["dimension"])
print("Total vectors:", stats["total_vector_count"])

In [ ]:
import importlib
import app.rag.vectorstore.pinecone_store as pinecone_module

importlib.reload(pinecone_module)

store = pinecone_module.PineconeStore()

uploaded = store.upsert_documents(
    documents=all_chunks[:3],
    embeddings=vectors[:3]
)

print("Uploaded vectors:", uploaded)

In [ ]:
stats = store.check_connection()

print("Total vectors:", stats["total_vector_count"])

In [ ]:
uploaded = store.upsert_documents(
    documents=all_chunks,
    embeddings=vectors
)

print("Vectors processed:", uploaded)

In [ ]:
sample_id = all_chunks[0].metadata["chunk_id"]

result = store.index.fetch(
    ids=[sample_id],
    namespace="nexadesk-kb"
)

record = result.vectors.get(sample_id)

if record is None:
    print("Chunk not found.")
else:
    print("Chunk ID:", record.id)
    print("Source:", record.metadata.get("source"))
    print("Document type:", record.metadata.get("document_type"))
    print("Text:", record.metadata.get("text"))
    print("Vector dimensions:", len(record.values))

In [ ]:
import importlib
import app.rag.vectorstore.pinecone_store as pinecone_module

importlib.reload(pinecone_module)

store = pinecone_module.PineconeStore()

# Confirm stored vector count.
stats = store.check_connection()
print("Stored vectors:", stats["total_vector_count"])

# Embed the user question.
query = "Why does OAuth authentication fail after a server restore?"

query_vector = service.embed_query(query)

# Search Pinecone.
retrieved_docs = store.similarity_search(
    query_embedding=query_vector,
    top_k=3
)


for rank, doc in enumerate(retrieved_docs, start=1):

    print(f"\n{'=' * 65}")
    print("Rank:", rank)
    print("Score:", round(doc.metadata["similarity_score"], 4))
    print("Source:", doc.metadata.get("source"))
    print("Document Type:", doc.metadata.get("document_type"))
    print("=" * 65)

    print(doc.page_content)
    

In [ ]:
import importlib
import app.rag.retrieval.bm25_retriever as bm25_module

importlib.reload(bm25_module)

bm25 = bm25_module.BM25Retriever(all_chunks)

query = "How do I fix NX-AUTH-4017?"

bm25_results = bm25.search(query, top_k=3)

for rank, doc in enumerate(bm25_results, start=1):

    print(f"\n{'=' * 65}")
    print("Rank:", rank)
    print("BM25 Score:", round(doc.metadata["bm25_score"], 4))
    print("Source:", doc.metadata.get("source"))
    print("Document Type:", doc.metadata.get("document_type"))
    print("=" * 65)

    print(doc.page_content)

In [ ]:
query_vector = service.embed_query(query)

dense_results = store.similarity_search(
    query_embedding=query_vector,
    top_k=3
)

for rank, doc in enumerate(dense_results, start=1):
    print(
        rank,
        round(doc.metadata["similarity_score"], 4),
        doc.metadata.get("source"),
        doc.metadata.get("error_code"),
        doc.metadata.get("ticket_id")
    )

In [ ]:
import importlib
import app.rag.retrieval.bm25_retriever as bm25_module

importlib.reload(bm25_module)

bm25 = bm25_module.BM25Retriever(all_chunks)

query = "How do I fix NX-AUTH-4017?"

results = bm25.search(query, top_k=5)

for rank, doc in enumerate(results, start=1):
    print(
        rank,
        round(doc.metadata["bm25_score"], 4),
        doc.metadata.get("source"),
        doc.metadata.get("error_code"),
        doc.metadata.get("ticket_id")
    )

In [ ]:
import importlib
import app.rag.retrieval.hybrid_retriever as hybrid_module

importlib.reload(hybrid_module)

hybrid = hybrid_module.HybridRetriever(
    embedding_service=service,
    pinecone_store=store,
    bm25_retriever=bm25
)

query = "How do I fix NX-AUTH-4017?"

hybrid_results = hybrid.search(
    query=query,
    top_k=5,
    candidate_k=10
)

for rank, doc in enumerate(hybrid_results, start=1):
    print(f"\n{'=' * 60}")
    print("Rank:", rank)
    print("RRF Score:", round(doc.metadata["rrf_score"], 5))
    print("Source:", doc.metadata.get("source"))
    print("Error Code:", doc.metadata.get("error_code"))
    print("Ticket ID:", doc.metadata.get("ticket_id"))
    print("=" * 60)
    print(doc.page_content)

In [ ]:
import importlib
import app.rag.retrieval.reranker as reranker_module

importlib.reload(reranker_module)

reranker = reranker_module.CrossEncoderReranker()

In [ ]:
query = "How do I fix NX-AUTH-4017?"

hybrid_candidates = hybrid.search(
    query=query,
    top_k=10,
    candidate_k=15
)

reranked_results = reranker.rerank(
    query=query,
    documents=hybrid_candidates,
    top_k=5
)

for rank, doc in enumerate(reranked_results, start=1):

    print(f"\n{'=' * 65}")
    print("Rank:", rank)
    print("RRF Score:", round(doc.metadata["rrf_score"], 5))
    print("Rerank Score:", round(doc.metadata["rerank_score"], 4))
    print("Source:", doc.metadata.get("source"))
    print("Error Code:", doc.metadata.get("error_code"))
    print("Ticket ID:", doc.metadata.get("ticket_id"))
    print("=" * 65)

    print(doc.page_content)

In [ ]:
print("BEFORE RERANKING")
for rank, doc in enumerate(hybrid_candidates[:5], 1):
    print(
        rank,
        doc.metadata.get("error_code"),
        doc.metadata.get("ticket_id"),
        round(doc.metadata["rrf_score"], 5)
    )

print("\nAFTER RERANKING")
for rank, doc in enumerate(reranked_results, 1):
    print(
        rank,
        doc.metadata.get("error_code"),
        doc.metadata.get("ticket_id"),
        round(doc.metadata["rerank_score"], 4)
    )

In [ ]:
import importlib
import app.rag.evaluation.retrieval_metrics as metrics_module

importlib.reload(metrics_module)

retrieved_ids = ["A", "B", "C", "D", "E"]

relevant_ids = {"A", "C", "F"}

k = 5

print(
    "Precision@5:",
    metrics_module.precision_at_k(retrieved_ids, relevant_ids, k)
)

print(
    "Recall@5:",
    metrics_module.recall_at_k(retrieved_ids, relevant_ids, k)
)

print(
    "Reciprocal Rank:",
    metrics_module.reciprocal_rank(retrieved_ids, relevant_ids)
)

In [ ]:
for chunk in all_chunks:

    if chunk.metadata.get("error_code") == "NX-AUTH-4017":

        print(
            "Chunk ID:", chunk.metadata["chunk_id"],
            "| Source:", chunk.metadata.get("source"),
            "| Ticket:", chunk.metadata.get("ticket_id")
        )
        

In [ ]:
auth_relevant_ids = {
    chunk.metadata["chunk_id"]
    for chunk in all_chunks
    if chunk.metadata.get("error_code") == "NX-AUTH-4017"
}

In [ ]:
evaluation_query = "How do I fix NX-AUTH-4017?"

In [ ]:
hybrid_results = hybrid.search(
    query=evaluation_query,
    top_k=5,
    candidate_k=10
)

retrieved_ids = [
    doc.metadata["chunk_id"]
    for doc in hybrid_results
]

print(
    "Precision@5:",
    metrics_module.precision_at_k(
        retrieved_ids,
        auth_relevant_ids,
        5
    )
)

print(
    "Recall@5:",
    metrics_module.recall_at_k(
        retrieved_ids,
        auth_relevant_ids,
        5
    )
)

print(
    "Reciprocal Rank:",
    metrics_module.reciprocal_rank(
        retrieved_ids,
        auth_relevant_ids
    )
)

In [ ]:
evaluation_queries = [
    {
        "query_id": "Q1",
        "query": "How do I fix NX-AUTH-4017?"
    },
    {
        "query_id": "Q2",
        "query": "Why does login fail after a server restore?"
    },
    {
        "query_id": "Q3",
        "query": "How can I resolve API rate limiting and HTTP 429 errors?"
    },
    {
        "query_id": "Q4",
        "query": "Why are webhook deliveries failing with HTTP 503?"
    },
    {
        "query_id": "Q5",
        "query": "How do I fix an SSO audience mismatch?"
    },
    {
        "query_id": "Q6",
        "query": "What should I do when OAuth tokens are rejected after a host patch?"
    },
    {
        "query_id": "Q7",
        "query": "How do I troubleshoot duplicate support ticket notifications?"
    },
    {
        "query_id": "Q8",
        "query": "What should I check when API requests are being throttled?"
    },
    {
        "query_id": "Q9",
        "query": "How do I resolve NX-AUTH-4020?"
    },
    {
        "query_id": "Q10",
        "query": "What are the recommended steps for troubleshooting webhook delivery failures?"
    }
]

print("Total evaluation queries:", len(evaluation_queries))

import pandas as pd

chunk_reference = pd.DataFrame([
    {
        "chunk_number": i,
        "chunk_id": chunk.metadata["chunk_id"],
        "document_type": chunk.metadata.get("document_type"),
        "source": chunk.metadata.get("source"),
        "error_code": chunk.metadata.get("error_code"),
        "ticket_id": chunk.metadata.get("ticket_id"),
        "text_preview": chunk.page_content[:150]
    }
    for i, chunk in enumerate(all_chunks, start=1)
])

display(chunk_reference)

In [ ]:
def inspect_chunks(keyword: str):

    matches = [
        (i, chunk)
        for i, chunk in enumerate(all_chunks, start=1)
        if keyword.lower() in chunk.page_content.lower()
    ]

    for i, chunk in matches:

        print("=" * 70)
        print("Chunk Number:", i)
        print("Chunk ID:", chunk.metadata["chunk_id"])
        print("Source:", chunk.metadata.get("source"))
        print("Document Type:", chunk.metadata.get("document_type"))
        print("-" * 70)
        print(chunk.page_content)
        print()

    print("Total matching chunks:", len(matches))

In [ ]:
inspect_chunks("429")

In [ ]:
relevance_labels = {}

relevance_labels["Q1"] = {
    all_chunks[i - 1].metadata["chunk_id"]
    for i in [1, 15, 25, 36]
}
relevance_labels["Q2"] = {
    all_chunks[i - 1].metadata["chunk_id"]
    for i in [1, 36]
}
relevance_labels["Q3"] = {
    all_chunks[i - 1].metadata["chunk_id"]
    for i in [2, 9, 16]
}

print("Q3 relevant chunks:", len(relevance_labels["Q3"]))

print("Q2 relevant chunks:", len(relevance_labels["Q2"]))

print("Q1 relevant chunks:", len(relevance_labels["Q1"]))

In [ ]:
inspect_chunks("server restore")


In [ ]:
import json

evaluation_export = {
    "queries": evaluation_queries,
    "chunks": [
        {
            "chunk_number": i,
            "chunk_id": chunk.metadata["chunk_id"],
            "metadata": chunk.metadata,
            "text": chunk.page_content
        }
        for i, chunk in enumerate(all_chunks, start=1)
    ]
}

with open("retrieval_evaluation_data.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_export, f, indent=2, ensure_ascii=False)

print("Exported:", len(evaluation_export["queries"]), "queries")
print("Exported:", len(evaluation_export["chunks"]), "chunks")

In [ ]:
# Relevant chunk numbers selected from the knowledge base
relevant_chunk_numbers = {
    "Q1": [1, 8, 15, 25, 36, 39],
    "Q2": [1, 8, 15, 25, 36, 39],
    "Q3": [2, 9, 14, 16, 22, 32, 37, 40],
    "Q4": [3, 7, 10, 14, 17, 24, 27, 38, 41],
    "Q5": [6, 8, 13, 20, 30, 39],
    "Q6": [1, 8, 14, 15, 25, 36, 39],
    "Q7": [4, 11, 14, 18, 23, 29, 34, 42],
    "Q8": [2, 9, 14, 16, 22, 32, 37, 40],
    "Q9": [6, 13, 20],
    "Q10": [3, 7, 10, 14, 17, 24, 27, 31, 38, 41]
}

# Convert chunk numbers into actual chunk IDs
relevance_labels = {
    query_id: {
        all_chunks[number - 1].metadata["chunk_id"]
        for number in numbers
    }
    for query_id, numbers in relevant_chunk_numbers.items()
}

# Validate
for query_id, ids in relevance_labels.items():
    print(f"{query_id}: {len(ids)} relevant chunks")

print("\nTotal labeled queries:", len(relevance_labels))

In [ ]:
import pandas as pd


def evaluate_retriever(method_name, search_function):
    results = []

    for item in evaluation_queries:
        query_id = item["query_id"]
        query = item["query"]

        # Retrieve the top 5 documents
        documents = search_function(query)

        # Extract retrieved chunk IDs
        retrieved_ids = [
            doc.metadata["chunk_id"]
            for doc in documents
        ]

        # Get manually labeled relevant chunk IDs
        relevant_ids = relevance_labels[query_id]

        # Calculate evaluation metrics
        results.append({
            "query_id": query_id,
            "method": method_name,
            "precision_at_5": metrics_module.precision_at_k(
                retrieved_ids, relevant_ids, 5
            ),
            "recall_at_5": metrics_module.recall_at_k(
                retrieved_ids, relevant_ids, 5
            ),
            "reciprocal_rank": metrics_module.reciprocal_rank(
                retrieved_ids, relevant_ids
            )
        })

    return pd.DataFrame(results)

In [ ]:
# Dense retrieval using embeddings + Pinecone
def dense_search(query):
    query_embedding = service.embed_query(query)

    return store.similarity_search(
        query_embedding=query_embedding,
        top_k=5
    )


# Evaluate Dense Retrieval
dense_results = evaluate_retriever(
    method_name="Dense",
    search_function=dense_search
)

# Display results for all 10 queries
display(dense_results)

In [ ]:
# BM25 retrieval
def bm25_search(query):
    return bm25.search(query, top_k=5)


# Evaluate BM25
bm25_results = evaluate_retriever(
    method_name="BM25",
    search_function=bm25_search
)

display(bm25_results)

In [ ]:
def hybrid_search(query):
    return hybrid.search(query, top_k=5)

hybrid_results = evaluate_retriever(
    method_name="Hybrid",
    search_function=hybrid_search
)

display(hybrid_results)

In [ ]:
import inspect

print(type(reranker))
print([name for name in dir(reranker) if not name.startswith("_")])

In [ ]:
import inspect

print(inspect.signature(reranker.rerank))

In [ ]:
# Hybrid Retrieval + Cross-Encoder Reranking
def hybrid_rerank_search(query):

    # Step 1: Retrieve 10 candidate documents
    candidates = hybrid.search(
        query,
        top_k=10
    )

    # Step 2: Rerank candidates and select the best 5
    reranked_documents = reranker.rerank(
        query=query,
        documents=candidates,
        top_k=5
    )

    return reranked_documents


# Step 3: Evaluate using our existing function
rerank_results = evaluate_retriever(
    method_name="Hybrid + Rerank",
    search_function=hybrid_rerank_search
)

display(rerank_results)

In [ ]:
# Combine evaluation results
all_results = pd.concat(
    [
        dense_results,
        bm25_results,
        hybrid_results,
        rerank_results
    ],
    ignore_index=True
)

# Calculate average metrics for each retrieval method
evaluation_summary = (
    all_results
    .groupby("method", as_index=False)
    .agg(
        mean_precision_at_5=("precision_at_5", "mean"),
        mean_recall_at_5=("recall_at_5", "mean"),
        mrr_at_5=("reciprocal_rank", "mean")
    )
    .round(4)
)

display(evaluation_summary)

In [ ]:
from app.rag.generation.context_builder import build_context

query = "How do I fix NX-AUTH-4017?"

# Retrieve candidate chunks
candidates = hybrid.search(query, top_k=10)

# Rerank and select the best 5
retrieved_docs = reranker.rerank(
    query=query,
    documents=candidates,
    top_k=5
)

# Build LLM context
context = build_context(retrieved_docs)

print(context)

In [ ]:
from app.rag.generation.prompt_builder import build_rag_prompt

prompt_template = build_rag_prompt()

# Reuse the context generated in Step 1
messages = prompt_template.format_messages(
    context=context,
    question=query
)

for message in messages:
    print(f"\n{message.type.upper()}:\n")
    print(message.content)

In [ ]:
from langchain_groq import ChatGroq
from app.core.config import settings

llm = ChatGroq(
    model='openai/gpt-oss-120b',
    api_key=settings.groq_api_key,
    temperature=0
)

response = llm.invoke(
    "Reply with exactly: Groq connection successful"
)

print(response.content)

In [ ]:
import importlib
import app.core.config as config

importlib.reload(config)

settings = config.Settings()

print("Model loaded:", settings.groq_model)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=settings.groq_model,
    api_key=settings.groq_api_key,
    temperature=0
)

response = llm.invoke("Reply with exactly: Groq service working")

print(response.content)

In [ ]:
from app.rag.generation.context_builder import build_context
from app.rag.generation.prompt_builder import build_rag_prompt
from app.rag.generation.llm_service import get_llm

# Step 1: Prepare retrieved context
context = build_context(retrieved_docs)

# Step 2: Prepare augmented prompt
prompt_template = build_rag_prompt()

messages = prompt_template.format_messages(
    context=context,
    question=query
)

# Step 3: Initialize Groq LLM
llm = get_llm()

# Step 4: Generate answer
response = llm.invoke(messages)

print(response.content)

In [ ]:
from app.rag.generation.rag_chain import generate_answer

result = generate_answer(
    question=query,
    retrieved_docs=retrieved_docs
)

print("QUESTION:\n", result["question"])
print("\nANSWER:\n", result["answer"])
print("\nSOURCES:\n", result["sources"])

In [ ]:
from app.rag.generation.rag_pipeline import run_rag_pipeline

result = run_rag_pipeline(
    question="How do I fix NX-AUTH-4017?",
    hybrid_retriever=hybrid,
    reranker=reranker
)

print("ANSWER:\n")
print(result["answer"])

print("\nSOURCES:\n")
for source in result["sources"]:
    print(source["citation"], source["source"], source["ticket_id"])

In [ ]:
result = run_rag_pipeline(
    question="How do I fix NX-AUTH-4017?",
    hybrid_retriever=hybrid,
    reranker=reranker
)

print(result["answer"])

In [ ]:
generation_eval_record = {
    "query_id": "Q1",
    "question": "How do I fix NX-AUTH-4017?",
    "reference_answer": (
        "NX-AUTH-4017 occurs when the application server clock "
        "is out of sync with the identity provider. "
        "Synchronize the application host using NTP, verify "
        "the server time, and restart the authentication worker."
    ),
    "generated_answer": result["answer"],
    "retrieved_sources": result["sources"]
}

print("Question:", generation_eval_record["question"])
print("\nReference answer:", generation_eval_record["reference_answer"])
print("\nGenerated answer:", generation_eval_record["generated_answer"])

In [ ]:
from app.rag.generation.context_builder import build_context
from app.rag.evaluation.generation_metrics import evaluate_faithfulness

context = build_context(retrieved_docs)

evaluation = evaluate_faithfulness(
    answer=result["answer"],
    context=context
)

print("Faithfulness:", evaluation["faithfulness"])
print("Supported:", evaluation["supported_claims"])
print("Total claims:", evaluation["total_claims"])

for claim in evaluation["claims"]:
    print("\nClaim:", claim["claim"])
    print("Supported:", claim["supported"])
    print("Reason:", claim["reason"])

In [ ]:
import importlib
import app.rag.evaluation.generation_metrics as gm

# Reload the updated Python file
importlib.reload(gm)

# Import the newly added function
from app.rag.evaluation.generation_metrics import evaluate_answer_relevance

In [ ]:
from app.rag.evaluation.generation_metrics import evaluate_answer_relevance

relevance_result = evaluate_answer_relevance(
    question=result["question"],
    answer=result["answer"]
)

print("Answer relevance:", relevance_result["answer_relevance"])
print("Reason:", relevance_result["reason"])

In [ ]:
import importlib
import app.rag.evaluation.generation_metrics as gm

importlib.reload(gm)

correctness_result = gm.evaluate_answer_correctness(
    question=generation_eval_record["question"],
    reference_answer=generation_eval_record["reference_answer"],
    generated_answer=generation_eval_record["generated_answer"]
)

print("Answer correctness:", correctness_result["answer_correctness"])
print("Reason:", correctness_result["reason"])

In [ ]:
import importlib
import app.rag.evaluation.generation_metrics as gm

importlib.reload(gm)

citation_result = gm.evaluate_citation_validity(
    answer=result["answer"],
    sources=result["sources"]
)

print("Citation validity:", citation_result["citation_validity"])
print("Cited sources:", citation_result["cited_sources"])
print("Valid citations:", citation_result["valid_citations"])
print("Invalid citations:", citation_result["invalid_citations"])

In [ ]:
import importlib
import app.rag.evaluation.generation_metrics as gm

importlib.reload(gm)

evaluation_report = gm.evaluate_generation(
    question=generation_eval_record["question"],
    reference_answer=generation_eval_record["reference_answer"],
    generated_answer=generation_eval_record["generated_answer"],
    context=context,
    sources=generation_eval_record["retrieved_sources"]
)

print("GENERATION EVALUATION REPORT")
print("-" * 35)

for metric in [
    "faithfulness",
    "answer_relevance",
    "answer_correctness",
    "citation_validity"
]:
    print(f"{metric}: {evaluation_report[metric]:.2%}")

In [ ]:
from pathlib import Path

file_path = Path("../app/rag/evaluation/generation_eval_questions.json")

print("File exists:", file_path.exists())

In [ ]:
import json
import pandas as pd

with open(file_path, "r", encoding="utf-8") as f:
    evaluation_dataset = json.load(f)

df_eval = pd.DataFrame(evaluation_dataset)

print("Total questions:", len(df_eval))
display(df_eval[["query_id", "question", "reference_answer"]])

In [ ]:
import time
import re
from groq import RateLimitError


def evaluate_with_retry(
    question,
    reference_answer,
    generated_answer,
    context,
    sources,
    max_retries=8
):
    for attempt in range(max_retries):
        try:
            return evaluate_generation(
                question=question,
                reference_answer=reference_answer,
                generated_answer=generated_answer,
                context=context,
                sources=sources
            )

        except RateLimitError as error:
            if attempt == max_retries - 1:
                raise

            # Extract Groq's suggested retry duration
            match = re.search(
                r"try again in ([\d.]+)s",
                str(error),
                flags=re.IGNORECASE
            )

            wait_seconds = (
                float(match.group(1)) + 2
                if match else 30
            )

            print(
                f"Rate limit reached. "
                f"Waiting {wait_seconds:.1f} seconds..."
            )

            time.sleep(wait_seconds)

In [ ]:
import pandas as pd

from app.rag.generation.rag_pipeline import run_rag_pipeline
from app.rag.generation.context_builder import build_context
from app.rag.evaluation.generation_metrics import evaluate_generation

evaluation_results = []

for _, row in df_eval.iterrows():

    question = row["question"]
    reference_answer = row["reference_answer"]

    print(f"Evaluating {row['query_id']}: {question}")

    # Step 1: Retrieve relevant chunks
    candidates = hybrid.search(
        question,
        top_k=10
    )

    # Step 2: Rerank and select top 5
    retrieved_docs = reranker.rerank(
        query=question,
        documents=candidates,
        top_k=5
    )

    # Step 3: Generate answer using the same chunks
    from app.rag.generation.rag_chain import generate_answer

    result = generate_answer(
        question=question,
        retrieved_docs=retrieved_docs
    )

    # Step 4: Build the exact context used for generation
    context = build_context(retrieved_docs)

    # Step 5: Evaluate generated answer
    metrics = evaluate_with_retry(
        question=question,
        reference_answer=reference_answer,
        generated_answer=result["answer"],
        context=context,
        sources=result["sources"]
    )

    evaluation_results.append({
        "query_id": row["query_id"],
        "question": question,
        "generated_answer": result["answer"],
        "faithfulness": metrics["faithfulness"],
        "answer_relevance": metrics["answer_relevance"],
        "answer_correctness": metrics["answer_correctness"],
        "citation_validity": metrics["citation_validity"]
    })

# Convert evaluation results to DataFrame
df_generation_results = pd.DataFrame(evaluation_results)

display(df_generation_results[
    [
        "query_id",
        "faithfulness",
        "answer_relevance",
        "answer_correctness",
        "citation_validity"
    ]
])

In [ ]:
# Get Q8's evaluation record
q8 = df_generation_results[
    df_generation_results["query_id"] == "Q8"
].iloc[0]

# Get the verified reference answer
q8_reference = df_eval[
    df_eval["query_id"] == "Q8"
].iloc[0]

print("QUESTION:")
print(q8["question"])

print("\nREFERENCE ANSWER:")
print(q8_reference["reference_answer"])

print("\nGENERATED ANSWER:")
print(q8["generated_answer"])

In [ ]:
q8_faithfulness = evaluate_faithfulness(
    answer=q8["generated_answer"],
    context=build_context(
        reranker.rerank(
            query=q8["question"],
            documents=hybrid.search(q8["question"], top_k=10),
            top_k=5
        )
    )
)

from pprint import pprint
pprint(q8_faithfulness)

In [114]:
from app.rag.generation.rag_pipeline import run_rag_pipeline

q8_result = run_rag_pipeline(
    question=q8["question"],
    hybrid_retriever=hybrid,
    reranker=reranker
)

print(q8_result["answer"])

The throttling is most likely caused by exceeding the tenant’s burst‑rate limit (120 requests / minute for standard tenants) — the platform returns **HTTP 429** when the limit is hit 【Source 1】.  

**Resolution steps**

1. **Lower request concurrency** – send fewer parallel calls so the total per‑minute count stays below 120.  
2. **Honor the `Retry‑After` header** – when a 429 is received, pause for the time indicated before retrying.  
3. **Add exponential backoff with jitter** – on each retry increase the wait time (e.g., 2ⁿ × base) and add a random jitter to avoid synchronized bursts.  
4. **Cap the maximum retry attempts** – prevent workers from retrying immediately in a tight loop (as seen in ticket SUP‑1889) 【Source 4】.  

These changes align with the documented best practice for handling NexaDesk API throttling and have resolved similar cases (e.g., tickets SUP‑1847 and SUP‑1889) 【Source 2】【Source 5】.
